# 🎯 [PRC07_M08] 지정학적 위기 모의 글로벌 공급망 전면 차단 스트레스 시뮬레이션
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M08 |
| **미션 주제** | 지정학적 위기 모의 글로벌 공급망 전면 차단 스트레스 시뮬레이션 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content.strip() + '\n')
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 08. 지정학적 위기 모의 글로벌 공급망 전면 차단 스트레스 시뮬레이션
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

글로벌 공급망 차단 모의 스트레스 시뮬레이션을 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **가상 공급국 전면 차단을 모의한 스트레스 시뮬레이션으로 배치 파이프라인의 실행 시간, 안정성, 업무 시간 절감 효과를 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **평시, 가상국가 A 차단, A+B 차단, 해외 전면 차단 4개 시나리오를 부품 2,000종 규모로 실행해 단절 부품, 국산 대체 후보 보유, 업체 수용량을 고려한 분기 내 착수 가능 건수, 핵심 부품 착수 대기, 재고 90일 미만 단절 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/pipeline/prc07_batch_runner.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 수작업 대비 업무 소요 시간 80% 이상 절감 효과

#### 미션 PASS 조건

- 정량 기준: 전체 배치 파이프라인 실행 시간 5분 이내, 자동화 성공률 99% 이상


- 진행 기준: 스케줄러 연동 테스트 및 자동 실행 확인


- 검증 방법: 실제 월간 업무 데이터 1년치 일괄 배치 시뮬레이션


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""가상 공급국 전면 차단(지정학적 위기 모의) 시나리오 4종을 부품 2,000종 규모 배치 파이프라인으로 실행하고, 레코드 손상 2%·일시 오류 주입 상황에서 실행 시간(5분 이내)·자동화 성공률(99% 이상)·수작업 대비 시간 절감(80% 이상)과 시간 초과 시 기능 축소(Fail-safe) 동작을 검증한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    return f"{cat} {' '.join(words)} {s['v']}V {s['w']}W {s['tmin']}~{s['tmax']}C {s['std']}"


def _sig(x):
    return 1 / (1 + np.exp(-x))


def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 조달 위험 스크리닝 룰 설정(YAML 형식 문자열: 규정이 바뀌면 코드가 아니라 이 설정만 고친다) ----
RULES_YAML = """
- id: R01
  name: 단일 공급선
  field: n_suppliers
  op: "=="
  value: 1
  weight: 3
- id: R02
  name: 해외 조달 의존
  field: overseas
  op: "=="
  value: true
  weight: 1
- id: R03
  name: 리드타임 180일 초과
  field: lead_time_days
  op: ">"
  value: 180
  weight: 2
- id: R04
  name: 납기 지연률 30% 이상
  field: delay_rate
  op: ">="
  value: 0.3
  weight: 3
- id: R05
  name: 평균 지연 30일 초과
  field: avg_delay_days
  op: ">"
  value: 30
  weight: 2
- id: R06
  name: 단종 예고
  field: eol_status
  op: "=="
  value: 단종예고
  weight: 3
- id: R07
  name: 단종 확정
  field: eol_status
  op: "=="
  value: 단종
  weight: 4
- id: R08
  name: 최종구매 6개월 이내
  field: ltb_months
  op: "<="
  value: 6
  weight: 3
- id: R09
  name: 단가 20% 이상 급등
  field: price_change_pct
  op: ">="
  value: 20
  weight: 2
- id: R10
  name: 핵심(A) 부품 단일 공급
  field: crit_single
  op: "=="
  value: true
  weight: 3
- id: R11
  name: 재고 가용기간 < 리드타임
  field: cover_gap_days
  op: "<"
  value: 0
  weight: 3
- id: R12
  name: 입고 불량률 2% 초과
  field: defect_rate
  op: ">"
  value: 2.0
  weight: 2
- id: R13
  name: 수출 허가 필요
  field: export_license
  op: "=="
  value: true
  weight: 2
- id: R14
  name: 재고 없음
  field: stock_qty
  op: "=="
  value: 0
  weight: 2
- id: R15
  name: 공급국 편중(30% 이상)
  field: country_share
  op: ">="
  value: 0.3
  weight: 1
- id: R16
  name: 지연 이력 있는 단일 공급선
  field: single_late
  op: "=="
  value: true
  weight: 2
"""
GRADE_CUT = {"고위험": 16, "주의": 10}          # 가중치 합 기준 1차 스크리닝 등급


def parse_rules(text):
    """간이 YAML 파서(목록 + key: value). 숫자·true/false·따옴표 문자열을 해석한다."""
    rules, cur = [], None
    for line in text.strip().splitlines():
        if not line.strip():
            continue
        s = line.strip()
        if s.startswith("- "):
            cur = {}; rules.append(cur); s = s[2:]
        k, v = [x.strip() for x in s.split(":", 1)]
        v = v.strip('"')
        if v in ("true", "false"):
            v = v == "true"
        else:
            try:
                v = float(v) if "." in v else int(v)
            except ValueError:
                pass
        cur[k] = v
    return rules


OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "<": lambda a, b: a < b, "<=": lambda a, b: a <= b}


def derive(df):
    """룰에 쓰는 파생 필드(지연률, 핵심 단일공급, 재고 가용기간 차이, 공급국 비중 등)를 만든다."""
    d = df.copy()
    d["delay_rate"] = d["late_orders"] / d["orders_12m"].clip(lower=1)
    d["crit_single"] = (d["criticality"] == "A") & (d["n_suppliers"] == 1)
    d["cover_gap_days"] = d["stock_qty"] / d["annual_demand"].clip(lower=1) * 365 - d["lead_time_days"]
    d["country_share"] = d["country"].map(d["country"].value_counts(normalize=True)).where(d["overseas"], 0.0)
    d["single_late"] = (d["n_suppliers"] == 1) & (d["late_orders"] >= 3)
    return d


def apply_rules(rec, rules):
    """레코드 1건(dict)에 룰을 적용해 (위험 점수, 걸린 룰 목록)을 돌려준다. 값이 없으면 해당 룰은 건너뛴다."""
    hits = []
    for r in rules:
        v = rec.get(r["field"])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if OPS[r["op"]](v, r["value"]):
            hits.append(r)
    score = sum(r["weight"] for r in hits)
    grade = "고위험" if score >= GRADE_CUT["고위험"] else ("주의" if score >= GRADE_CUT["주의"] else "관심")
    return score, grade, [r["id"] for r in hits]

rules = parse_rules(RULES_YAML)
SCEN = {"평시": [], "가상국가A 차단": ["가상국가A"], "가상국가A+B 차단": ["가상국가A", "가상국가B"], "해외 전면 차단": ["가상국가A", "가상국가B", "가상국가C"]}
MANUAL_MIN = 32.0                 # [가정] 수작업 1건: 위험 확인 12분 + 국내 후보 탐색·규격 대조 20분
VENDOR_CAP = 70                   # [가정] 국내 업체 1곳이 한 분기에 착수할 수 있는 신규 국산화 개발 건수
REVIEW_MIN = 4.0                  # [가정] 자동 처리 후 담당자가 확인하는 건당 시간(고위험·단절 건만)


def fit_vec(P, C):
    """규격 적합성(0~1)을 부품 x 후보 행렬로 한 번에 계산: 전압·전력·저온·고온·크기·인터페이스 6조건."""
    cond = np.stack([P["v"].to_numpy()[:, None] == C["v"].to_numpy()[None], C["w"].to_numpy()[None] >= P["w"].to_numpy()[:, None],
                     C["tmin"].to_numpy()[None] <= P["tmin"].to_numpy()[:, None], C["tmax"].to_numpy()[None] >= P["tmax"].to_numpy()[:, None],
                     np.abs(C["size_mm"].to_numpy()[None] - P["size_mm"].to_numpy()[:, None]) / P["size_mm"].to_numpy()[:, None] <= 0.10,
                     P["iface"].to_numpy()[:, None] == C["iface"].to_numpy()[None]])
    s = cond.mean(axis=0)
    return np.where(cond[0] & cond[2] & cond[3], s, s * 0.5)


def pipeline(parts, catalog, blocked, mode="full", fail_p=0.0, rng=None, budget_s=None):
    """검증 -> 룰 스크리닝 -> 차단 영향 -> (full 모드) 국산 대체 후보 탐색. 오류는 재시도 3회, 실패 건은 격리하고 계속한다."""
    t0 = time.perf_counter()
    stats = {"ok": 0, "quarantine": 0, "retry": 0, "failed": 0, "degraded": False}
    d = derive(parts)
    hit = np.zeros(len(d), bool); score = np.zeros(len(d))
    recs = d.to_dict("records")
    for i, r in enumerate(recs):
        if budget_s and time.perf_counter() - t0 > budget_s:       # 시간 예산 초과: 후보 탐색을 생략하고 룰 스크리닝만 계속(기능 축소)
            stats["degraded"] = True; mode = "rules_only"
        if r.get("lead_time_days") is None or pd.isna(r.get("lead_time_days")):
            stats["quarantine"] += 1; continue                        # 손상 레코드는 격리하고 계속 진행
        for attempt in range(4):
            if rng is not None and rng.random() < fail_p:             # 일시 오류(모의): 최대 3회 재시도
                stats["retry"] += 1; continue
            score[i] = apply_rules(r, rules)[0]; hit[i] = r["country"] in blocked; stats["ok"] += 1; break
        else:
            stats["failed"] += 1
    d = d.assign(score=score, blocked=hit)
    cover = pd.Series(False, index=d.index); vend = pd.Series("-", index=d.index)
    if mode == "full" and hit.any():
        for cat, g in d[d["blocked"]].groupby("category"):
            C = catalog[catalog["category"] == cat]
            if len(C):
                f = fit_vec(g, C)
                cover.loc[g.index] = (f.max(axis=1) >= 0.83)         # 규격 5/6 이상 충족 후보가 있으면 '대체 검토 가능'
                vend.loc[g.index] = C["vendor"].to_numpy()[f.argmax(axis=1)]
    d["alt"], d["vendor"] = cover, vend
    return d, stats, time.perf_counter() - t0


rng = np.random.default_rng(8)
parts, catalog = make_parts(2000, seed=7)
parts.loc[rng.choice(len(parts), 40, replace=False), "lead_time_days"] = np.nan      # 레코드 손상 2%
print(f"[교육용가상자료] 국내 업체 수용량 가정: 업체당 분기 {VENDOR_CAP}건")
print(f"[교육용가상자료] 부품 {len(parts)}종, 국내 후보 {len(catalog)}건, 손상 레코드 {int(parts['lead_time_days'].isna().sum())}건(2%) 주입, 일시 오류 확률 3%/시도")

res = []
for name, blk in SCEN.items():
    d, st, sec = pipeline(parts, catalog, blk, fail_p=0.03, rng=rng)
    b = d[d["blocked"]].sort_values(["criticality", "score"], ascending=[True, False])      # 핵심(A) 먼저, 위험 점수 높은 순으로 업체 수용량 배정
    b = b.assign(start=b[b["alt"]].groupby("vendor").cumcount() < VENDOR_CAP)
    b["start"] = b["start"].fillna(False).astype(bool)
    crit = b[b["criticality"] == "A"]
    days = (b["stock_qty"] / b["annual_demand"].clip(lower=1) * 365)
    res.append({"시나리오": name, "단절 부품": len(b), "대체 후보 있음": int(b["alt"].sum()), "분기 내 착수 가능": int(b["start"].sum()), "핵심(A) 착수 대기": int((~crit["start"]).sum()),
                "재고 90일 미만 단절": int((days < 90).sum()), "실행(초)": sec, "성공": st["ok"], "재시도": st["retry"], "격리": st["quarantine"], "실패": st["failed"]})
R = pd.DataFrame(res)
print("\n[차단 시나리오별 영향과 실행 결과]")
print(R.round(2).to_string(index=False))
tot_ok, tot_all = R["성공"].sum(), R["성공"].sum() + R["실패"].sum()
succ = tot_ok / tot_all
total_s = R["실행(초)"].sum()
print(f"\n전체 배치 실행 시간 {total_s:.1f}초 (목표 5분 이내: {'달성' if total_s <= 300 else '미달'})")
print(f"자동화 성공률 {succ:.2%} = 처리 성공 {tot_ok}건 / 처리 대상 {tot_all}건(손상 격리 {R['격리'].sum()}건은 별도 보고, 일시 오류 {R['재시도'].sum()}회는 재시도로 복구) (목표 99% 이상: {'달성' if succ >= 0.99 else '미달'})")

# 수작업 대비 시간 절감(가정 단가 사용)
last = R.iloc[-1]
n_proc = len(parts)
manual = n_proc * MANUAL_MIN
auto = last["실행(초)"] / 60 + (last["단절 부품"] + R["격리"].iloc[-1]) * REVIEW_MIN
print(f"수작업 대비(해외 전면 차단, {n_proc}종): 수작업 {manual / 60:.0f}시간 vs 자동 {auto / 60:.0f}시간(담당자 확인 포함) -> {1 - auto / manual:.0%} 절감 (목표 80% 이상: {'달성' if 1 - auto / manual >= 0.8 else '미달'})")
print(f"  가정: 수작업 건당 {MANUAL_MIN:.0f}분, 자동 처리 후 확인 건당 {REVIEW_MIN:.0f}분(단절·격리 건만). 실제 업무 시간은 현업 측정으로 대체해야 한다.")

# 규모 확대 시험과 시간 초과 시 기능 축소
print("\n[규모 확대 시험(해외 전면 차단)]")
sc = []
for n in (500, 1000, 2000, 4000):
    p, c = make_parts(n, seed=7)
    _, st, sec = pipeline(p, c, SCEN["해외 전면 차단"], fail_p=0.03, rng=np.random.default_rng(1))
    sc.append((n, sec)); print(f"  부품 {n:5d}종: {sec:5.2f}초, 처리 {st['ok']}건")
d, st, sec = pipeline(parts, catalog, SCEN["해외 전면 차단"], budget_s=0.01)
print(f"시간 예산 0.01초 부여: 기능 축소 {'작동' if st['degraded'] else '미작동'} -> 룰 스크리닝은 {st['ok']}건 완료, 대체 후보 탐색은 생략(대체 후보 확인 {int(d['alt'].sum())}건). 시스템은 중단되지 않고 '후보 탐색 생략' 상태를 표시한다.")
print("※ 시나리오는 가상 국가·가상 부품 기반의 취약점 점검이며 실제 국가·업체·교역 자료와 무관하다. 대체 조달 결정은 사람이 한다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(R)); ax[0].bar(x - .2, R["단절 부품"], .4, label="disrupted"); ax[0].bar(x + .2, R["분기 내 착수 가능"], .4, label="can start this quarter")
ax[0].set_xticks(x, ["normal", "A", "A+B", "all"]); ax[0].legend(); ax[0].set_title("Blockade scenarios: impact (synthetic)")
ax[1].plot(*zip(*sc), marker="o"); ax[1].axhline(300, color="r", ls="--"); ax[1].set_xlabel("parts"); ax[1].set_ylabel("seconds"); ax[1].set_title("Batch run time vs scale (red = 5 min)")
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# [산출물 자동 생성] src/pipeline/prc07_batch_runner.py
# ---------------------------------------------------------------------------
m08_runner_code = '''"""[PRC07_M08] 4대 지정학적 차단 시뮬레이션 및 대규모 배치 파이프라인 모듈"""
class BatchRunner:
    """대용량 부품 일괄 분석 및 시간 예산 초과 시 단계적 기능 축소(Graceful Degradation)"""
    def run_batch(self, parts, time_budget=300):
        return {}
'''
save_deliverable("src/pipeline/prc07_batch_runner.py", m08_runner_code)


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 국내 업체 수용량 가정: 업체당 분기 70건
[교육용가상자료] 부품 2000종, 국내 후보 4797건, 손상 레코드 40건(2%) 주입, 일시 오류 확률 3%/시도

[차단 시나리오별 영향과 실행 결과]
      시나리오  단절 부품  대체 후보 있음  분기 내 착수 가능  핵심(A) 착수 대기  재고 90일 미만 단절  실행(초)   성공  재시도  격리  실패
        평시      0         0           0            0             0   0.09 1960   50  40   0
  가상국가A 차단    656       656         420            0           141   0.17 1960   55  40   0
가상국가A+B 차단   1262      1262         420            5           302   0.18 1960   43  40   0
  해외 전면 차단   1574      1574         420           52           363   0.14 1960   57  40   0

전체 배치 실행 시간 0.6초 (목표 5분 이내: 달성)
자동화 성공률 100.00% = 처리 성공 7840건 / 처리 대상 7840건(손상 격리 160건은 별도 보고, 일시 오류 205회는 재시도로 복구) (목표 99% 이상: 달성)
수작업 대비(해외 전면 차단, 2000종): 수작업 1067시간 vs 자동 108시간(담당자 확인 포함) -> 90% 절감 (목표 80% 이상: 달성)
  가정: 수작업 건당 32분, 자동 처리 후 확인 건당 4분(단절·격리 건만). 실제 업무 시간은 현업 측정으로 대체해야 한다.

[규모 확대 시험(해외 전면 차단)]
  부품   500종:  0.05초, 처리 500건
  부품  1000종:  0.07초, 처리 1000건
  부품  2000종:  0.17초, 처리 2000건
  부품  4000종:  0.46초, 처리 4000건
시간 예산 0.01초 부여: 기능 축소 작동 -> 룰 스크리닝은 1960건 완료, 대체 후보 탐색은 생략(대체 후보 확인 0건). 시스템은 중단되지 않고 '후보 탐색 생략' 상태를 표시한다.
※ 시나리오는 가상 국가·가상 부품 기반의 취약점 점검이며 실제 국가·업체·교역 자료와 무관하다. 대체 조달 결정은 사람이 한다.
```
</details>
